# Обучение модели: отклик на кредитный оффер

Полный пайплайн: EDA → признаки → подбор гиперпараметров Optuna → ансамбль XGBoost + CatBoost → анализ результатов. Метрика — ROC-AUC.

## 1. Разведочный анализ (EDA)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna

pd.set_option("display.max_columns", None)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
train = pd.read_csv("train_apps.csv")
test = pd.read_csv("test_apps.csv")
sample_submission = pd.read_csv("sample_submission.csv")
pd.DataFrame(
    {"rows": [len(train), len(test), len(sample_submission)],
     "columns": [train.shape[1], test.shape[1], sample_submission.shape[1]]},
    index=["train", "test", "sample_submission"],
)

,rows,columns
train,145241,28
test,36311,27
sample_submission,45032,2


In [7]:
train.sample(10)

,front_id,decision_day,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,corp_credit_products,sum_deb_ul_90,sum_deb_ul_30,cnt_deb_loan_90,cnt_deb_ul_ip_90,cnt_deb_ul_ip_30,balance_rur_amt_30_min,cnt_cred_loan_90,loan_rev_max_start_non_fin,loan_rev_min_start_fin,app_term_mean_360,overdraft_app_term_max_360,days_from_authperson_registration,fl_hdb_bki_total_active_products,corp_list,count_all_corp_dashboard_events,p75_time_spent_minutes,sum_deb_investment_90,db_group_last,fl_adminarea,target_value
140763,135397,2025-04-29,-0.623060,-1.561044,1.151140,0.709770,1.602779,0.000000,0.012443,0.836155,5.389124,-0.125126,0.022846,-5.561599,14.104880,NaN,2.280838,-3.092414,1.86909,1.531124,2.143483,-3.081808,-1.817031,-0.792560,NaN,overdraft,г. Санкт - Петербург,0
105107,230198,2024-09-23,-1.963050,-4.168248,-3.416908,3.193963,0.801389,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.328403,NaN,NaN,NaN,NaN,NaN,Псковская область,0
135204,151625,2025-03-25,0.845440,4.145626,3.141486,-1.064654,1.602779,0.000000,0.370716,2.473885,0.000000,2.406081,0.454029,-1.091507,0.000000,1.091644,NaN,-2.650641,NaN,-0.167370,0.000000,0.268287,2.706874,0.696299,0.004367,vkl,Свердловская область,0
45688,122695,2024-05-08,-1.963050,-3.778433,-3.109402,2.129309,-0.400695,0.000000,-2.136745,-14.231284,0.000000,-3.558315,-0.495469,-3.393063,0.000000,NaN,NaN,0.000000,NaN,-1.553440,-0.812825,-1.073782,-2.825349,-0.704883,NaN,inn_scoring,Краснодарский край,0
137990,129713,2025-04-10,1.690880,2.283353,1.672433,-1.064654,1.602779,1.954805,1.009241,5.834846,6.695054,2.420490,0.388648,-5.561599,19.717776,-1.367014,-1.727138,-1.767094,NaN,-1.622867,-1.351879,3.195430,0.310158,0.448186,1.240030,vkl,Республика Татарстан (Татарстан),0
18046,155382,2024-03-19,1.339991,2.862570,2.129347,-2.484194,-0.400695,1.690106,-0.274517,-0.223242,0.000000,-1.141415,0.069498,NaN,0.000000,NaN,NaN,-0.883547,NaN,NaN,-4.700790,1.214168,1.326321,0.261326,NaN,zalog_light,NaN,1
34159,222824,2024-04-16,1.339991,-2.780470,-2.322160,0.354885,-0.400695,0.000000,-0.052821,NaN,0.000000,0.017076,NaN,-1.643096,0.000000,NaN,NaN,NaN,NaN,-1.666259,-0.812825,0.755232,-0.803147,-0.075814,NaN,NaN,Ленинградская область,0
125256,181090,2025-01-21,-0.272171,2.067156,1.501886,-0.177442,1.602779,0.000000,0.487130,2.463047,3.953325,2.098785,0.296290,3.435362,0.000000,NaN,NaN,-2.429754,NaN,2.565097,-2.046841,0.233090,-5.505577,-0.796875,1.205443,inn_scoring,NaN,0
142357,105342,2025-05-15,2.185431,5.211760,3.982505,1.596982,1.602779,0.000000,0.543232,3.210645,0.000000,1.714824,0.385002,-5.557701,0.000000,NaN,-0.261611,-2.307044,NaN,4.046630,0.861630,1.888678,0.750977,0.222632,NaN,ebg,Свердловская область,0
69349,141469,2024-06-26,0.000000,-2.266626,-1.916815,-3.548848,-0.400695,0.000000,NaN,NaN,0.000000,-0.705855,0.143501,-5.561599,0.000000,NaN,NaN,NaN,NaN,NaN,-1.351879,-0.539163,-1.677178,-0.696524,NaN,NaN,Челябинская область,0


In [3]:
train["target_value"].value_counts(normalize=True).rename("share").to_frame()

,share
target_value,
0,0.939094
1,0.060906


In [4]:
train.isna().sum()

front_id                                  0
decision_day                              0
loan_amount_last                          0
overdraft_limit_min                       0
overdraft_limit_max                       0
offered_rate                              0
cb_rate                                   0
corp_credit_products                  51188
sum_deb_ul_90                         54111
sum_deb_ul_30                         61453
cnt_deb_loan_90                       31458
cnt_deb_ul_ip_90                      30305
cnt_deb_ul_ip_30                      33373
balance_rur_amt_30_min                34826
cnt_cred_loan_90                      31458
loan_rev_max_start_non_fin           132635
loan_rev_min_start_fin               124706
app_term_mean_360                     55883
overdraft_app_term_max_360           139732
days_from_authperson_registration     78473
fl_hdb_bki_total_active_products      24368
corp_list                             51188
count_all_corp_dashboard_events 

In [4]:
missing_share = train.isna().mean()
missing_share[missing_share > 0].sort_values(ascending=False).to_frame("missing_share")

,missing_share
overdraft_app_term_max_360,0.962070
loan_rev_max_start_non_fin,0.913206
sum_deb_investment_90,0.886093
loan_rev_min_start_fin,0.858614
days_from_authperson_registration,0.540295
sum_deb_ul_30,0.423111
app_term_mean_360,0.384761
db_group_last,0.384761
sum_deb_ul_90,0.372560
p75_time_spent_minutes,0.352435


In [5]:
def missingness_lift(column):
    is_missing = train[column].isna()
    return pd.Series({
        "missing_share": is_missing.mean(),
        "target_if_present": train.loc[~is_missing, "target_value"].mean(),
        "target_if_missing": train.loc[is_missing, "target_value"].mean(),
    })

columns_with_missing = train.columns[train.isna().any()]
missingness_report = pd.DataFrame({c: missingness_lift(c) for c in columns_with_missing}).T
missingness_report.sort_values("target_if_present", ascending=False).head(8)

,missing_share,target_if_present,target_if_missing
overdraft_app_term_max_360,0.962070,0.309131,0.051119
loan_rev_max_start_non_fin,0.913206,0.203871,0.047318
loan_rev_min_start_fin,0.858614,0.164354,0.043871
days_from_authperson_registration,0.540295,0.091256,0.035082
sum_deb_ul_30,0.423111,0.090204,0.020959
sum_deb_investment_90,0.886093,0.090002,0.057165
sum_deb_ul_90,0.372560,0.085373,0.019700
corp_credit_products,0.352435,0.082071,0.022017


In [6]:
train_calendar = pd.to_datetime(train["decision_day"])
test_calendar = pd.to_datetime(test["decision_day"])
pd.DataFrame(
    {"start": [train_calendar.min().date(), test_calendar.min().date()],
     "end": [train_calendar.max().date(), test_calendar.max().date()]},
    index=["train", "test"],
)

,start,end
train,2024-02-01,2025-06-05
test,2025-06-05,2025-12-01


In [7]:
submission_ids = set(sample_submission["front_id"])
pd.Series({
    "submission_ids": len(submission_ids),
    "covered_by_test_features": len(submission_ids & set(test["front_id"])),
    "covered_by_train_rows": len(submission_ids & set(train["front_id"])),
})

submission_ids              45032
covered_by_test_features    36311
covered_by_train_rows        8721
dtype: int64

## 2. Признаки и конфигурация

In [8]:
ID_COLUMN = "front_id"
TARGET_COLUMN = "target_value"
DATE_COLUMN = "decision_day"
START_DATE = pd.Timestamp("2024-02-01")
HOLDOUT_MONTHS = 2
N_SPLITS = 5
N_OPTUNA_TRIALS = 30
RANDOM_STATE = 42

MISSINGNESS_BLOCKS = {
    "corp_block_isna": [
        "corp_credit_products",
        "corp_list",
        "count_all_corp_dashboard_events",
        "p75_time_spent_minutes",
    ],
    "deb_cred_loan_isna": ["cnt_deb_loan_90", "cnt_cred_loan_90"],
    "app_term_db_group_isna": ["app_term_mean_360", "db_group_last"],
}

SINGLE_MISSING_COLUMNS = [
    "sum_deb_ul_90",
    "sum_deb_ul_30",
    "cnt_deb_ul_ip_90",
    "cnt_deb_ul_ip_30",
    "balance_rur_amt_30_min",
    "loan_rev_max_start_non_fin",
    "loan_rev_min_start_fin",
    "overdraft_app_term_max_360",
    "days_from_authperson_registration",
    "fl_hdb_bki_total_active_products",
    "sum_deb_investment_90",
    "fl_adminarea",
]

CATEGORICAL_COLUMNS = ["db_group_last", "fl_adminarea"]

RARE_COLUMNS = [
    "overdraft_app_term_max_360",
    "loan_rev_max_start_non_fin",
    "loan_rev_min_start_fin",
    "sum_deb_investment_90",
]

ALL_MISSING_COLUMNS = [
    column for columns in MISSINGNESS_BLOCKS.values() for column in columns
] + SINGLE_MISSING_COLUMNS

XGB_FIXED_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "enable_categorical": True,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

CATBOOST_PARAMS = {
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 3.0,
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
}

In [9]:
def build_features(data):
    features = data.copy()

    parsed_date = pd.to_datetime(features[DATE_COLUMN])
    features["month"] = parsed_date.dt.month.astype("int16")
    features["dayofweek"] = parsed_date.dt.dayofweek.astype("int16")
    features["day"] = parsed_date.dt.day.astype("int16")
    features["days_since_start"] = (parsed_date - START_DATE).dt.days.astype("int32")

    features["n_missing"] = features[ALL_MISSING_COLUMNS].isna().sum(axis=1).astype("int16")
    features["n_present_rare"] = features[RARE_COLUMNS].notna().sum(axis=1).astype("int8")

    for indicator_name, block_columns in MISSINGNESS_BLOCKS.items():
        features[indicator_name] = features[block_columns[0]].isna().astype("int8")
    for column in SINGLE_MISSING_COLUMNS:
        features[f"{column}_isna"] = features[column].isna().astype("int8")

    features["loan_amount_vs_overdraft_limit"] = features["loan_amount_last"] - features["overdraft_limit_max"]
    features["offered_rate_spread"] = features["offered_rate"] - features["cb_rate"]
    features["overdraft_limit_range"] = features["overdraft_limit_max"] - features["overdraft_limit_min"]
    features["debit_ip_trend_30_90"] = features["cnt_deb_ul_ip_30"] - features["cnt_deb_ul_ip_90"]
    features["debit_sum_trend_30_90"] = features["sum_deb_ul_30"] - features["sum_deb_ul_90"]
    features["debit_vs_credit_loans"] = features["cnt_deb_loan_90"] - features["cnt_cred_loan_90"]
    features["offered_rate_clipped"] = features["offered_rate"].clip(-5, 5)

    for column in CATEGORICAL_COLUMNS:
        features[column] = features[column].fillna("missing").astype("category")

    columns_to_drop = [ID_COLUMN, DATE_COLUMN]
    if TARGET_COLUMN in features.columns:
        columns_to_drop.append(TARGET_COLUMN)
    return features.drop(columns=columns_to_drop)

In [10]:
def align_categories(train_features, test_features):
    for column in CATEGORICAL_COLUMNS:
        categories = train_features[column].cat.categories.union(test_features[column].cat.categories)
        train_features[column] = train_features[column].cat.set_categories(categories)
        test_features[column] = test_features[column].cat.set_categories(categories)
    return train_features, test_features[train_features.columns]


def with_string_categories(features):
    string_features = features.copy()
    for column in CATEGORICAL_COLUMNS:
        string_features[column] = string_features[column].astype(str)
    return string_features


def rank_average(*score_arrays):
    return sum(rankdata(scores) for scores in score_arrays) / len(score_arrays)


def fit_xgboost(train_features, train_target, n_estimators, validation=None):
    model = XGBClassifier(n_estimators=n_estimators, **XGB_PARAMS)
    if validation is None:
        model.fit(train_features, train_target, verbose=False)
    else:
        model.set_params(early_stopping_rounds=150)
        model.fit(train_features, train_target, eval_set=[validation], verbose=False)
    return model


def fit_catboost(train_features, train_target, iterations, validation=None):
    model = CatBoostClassifier(iterations=iterations, **CATBOOST_PARAMS)
    if validation is None:
        model.fit(train_features, train_target, cat_features=CATEGORICAL_COLUMNS)
    else:
        model.set_params(early_stopping_rounds=150)
        model.fit(train_features, train_target, cat_features=CATEGORICAL_COLUMNS, eval_set=validation)
    return model

In [11]:
target = train[TARGET_COLUMN]
train_dates = pd.to_datetime(train[DATE_COLUMN])

features = build_features(train)
test_features = build_features(test)
features, test_features = align_categories(features, test_features)
string_features = with_string_categories(features)
string_test_features = with_string_categories(test_features)

is_holdout = (train_dates > train_dates.max() - pd.DateOffset(months=HOLDOUT_MONTHS)).values
training_features, training_target = features[~is_holdout], target[~is_holdout]
training_string_features = string_features[~is_holdout]
holdout_features, holdout_target = features[is_holdout], target[is_holdout]
holdout_string_features = string_features[is_holdout]
features.shape

(145241, 53)

## 3. Подбор гиперпараметров XGBoost (Optuna)

In [12]:
def xgboost_objective(trial):
    params = {
        **XGB_FIXED_PARAMS,
        "max_depth": trial.suggest_int("max_depth", 4, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 8.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
    }
    model = XGBClassifier(n_estimators=2000, early_stopping_rounds=100, **params)
    model.fit(training_features, training_target, eval_set=[(holdout_features, holdout_target)], verbose=False)
    return roc_auc_score(holdout_target, model.predict_proba(holdout_features)[:, 1])


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=1))
study.optimize(xgboost_objective, n_trials=N_OPTUNA_TRIALS)
print("best holdout auc:", round(study.best_value, 5))
study.best_params

best holdout auc: 0.75744


{'max_depth': 7,
 'learning_rate': 0.034524580465644356,
 'min_child_weight': 24,
 'subsample': 0.6596976864847959,
 'colsample_bytree': 0.7901744864663485,
 'reg_lambda': 1.2693143089738832,
 'reg_alpha': 0.05288315751624517}

In [13]:
XGB_PARAMS = {**XGB_FIXED_PARAMS, **study.best_params}
XGB_PARAMS

{'objective': 'binary:logistic',
 'eval_metric': 'auc',
 'tree_method': 'hist',
 'enable_categorical': True,
 'n_jobs': -1,
 'random_state': 42,
 'max_depth': 7,
 'learning_rate': 0.034524580465644356,
 'min_child_weight': 24,
 'subsample': 0.6596976864847959,
 'colsample_bytree': 0.7901744864663485,
 'reg_lambda': 1.2693143089738832,
 'reg_alpha': 0.05288315751624517}

## 4. Валидация по времени

In [14]:
xgb_holdout = fit_xgboost(training_features, training_target, 3000, (holdout_features, holdout_target))
catboost_holdout = fit_catboost(training_string_features, training_target, 3000, (holdout_string_features, holdout_target))

xgb_rounds = xgb_holdout.best_iteration + 1
catboost_rounds = catboost_holdout.best_iteration_ + 1

xgb_holdout_scores = xgb_holdout.predict_proba(holdout_features)[:, 1]
catboost_holdout_scores = catboost_holdout.predict_proba(holdout_string_features)[:, 1]

pd.Series({
    "xgboost": roc_auc_score(holdout_target, xgb_holdout_scores),
    "catboost": roc_auc_score(holdout_target, catboost_holdout_scores),
    "ensemble": roc_auc_score(holdout_target, rank_average(xgb_holdout_scores, catboost_holdout_scores)),
}, name="holdout_auc").round(5)

xgboost     0.75744
catboost    0.75333
ensemble    0.75819
Name: holdout_auc, dtype: float64

## 5. ансамбль

In [15]:
folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
xgb_oof = np.zeros(len(features))
catboost_oof = np.zeros(len(features))

for train_index, valid_index in folds.split(features, target):
    xgb_fold = fit_xgboost(features.iloc[train_index], target.iloc[train_index], xgb_rounds)
    xgb_oof[valid_index] = xgb_fold.predict_proba(features.iloc[valid_index])[:, 1]

    catboost_fold = fit_catboost(string_features.iloc[train_index], target.iloc[train_index], catboost_rounds)
    catboost_oof[valid_index] = catboost_fold.predict_proba(string_features.iloc[valid_index])[:, 1]

xgb_oof_by_id = pd.Series(xgb_oof, index=train[ID_COLUMN].values)
catboost_oof_by_id = pd.Series(catboost_oof, index=train[ID_COLUMN].values)

pd.Series({
    "xgboost": roc_auc_score(target, xgb_oof),
    "catboost": roc_auc_score(target, catboost_oof),
    "ensemble": roc_auc_score(target, rank_average(xgb_oof, catboost_oof)),
}, name="oof_auc").round(5)

xgboost     0.83529
catboost    0.83506
ensemble    0.83673
Name: oof_auc, dtype: float64

In [16]:
xgb_model = fit_xgboost(features, target, int(xgb_rounds * 1.1))
catboost_model = fit_catboost(string_features, target, int(catboost_rounds * 1.1))

xgb_test_scores = pd.Series(xgb_model.predict_proba(test_features)[:, 1], index=test[ID_COLUMN].values)
catboost_test_scores = pd.Series(catboost_model.predict_proba(string_test_features)[:, 1], index=test[ID_COLUMN].values)

xgb_scores_by_id = pd.concat([xgb_test_scores, xgb_oof_by_id])
catboost_scores_by_id = pd.concat([catboost_test_scores, catboost_oof_by_id])
xgb_scores_by_id = xgb_scores_by_id[~xgb_scores_by_id.index.duplicated()]
catboost_scores_by_id = catboost_scores_by_id[~catboost_scores_by_id.index.duplicated()]

submission = sample_submission[[ID_COLUMN]].copy()
xgb_aligned = submission[ID_COLUMN].map(xgb_scores_by_id).fillna(xgb_scores_by_id.median())
catboost_aligned = submission[ID_COLUMN].map(catboost_scores_by_id).fillna(catboost_scores_by_id.median())

ensemble_scores = rank_average(xgb_aligned.values, catboost_aligned.values)
submission[TARGET_COLUMN] = (ensemble_scores - ensemble_scores.min()) / (ensemble_scores.max() - ensemble_scores.min())
submission.to_csv("submission.csv", index=False)
submission.head()

,front_id,target_value
0,238459,0.112138
1,151533,0.339312
2,195027,0.555980
3,195034,0.630846
4,129620,0.625471


## 6. Анализ результатов

In [17]:
validation_comparison = pd.DataFrame({
    "holdout_time": [
        roc_auc_score(holdout_target, xgb_holdout_scores),
        roc_auc_score(holdout_target, catboost_holdout_scores),
        roc_auc_score(holdout_target, rank_average(xgb_holdout_scores, catboost_holdout_scores)),
    ],
    "oof_random": [
        roc_auc_score(target, xgb_oof),
        roc_auc_score(target, catboost_oof),
        roc_auc_score(target, rank_average(xgb_oof, catboost_oof)),
    ],
}, index=["xgboost", "catboost", "ensemble"]).round(5)
validation_comparison

,holdout_time,oof_random
xgboost,0.75744,0.83529
catboost,0.75333,0.83506
ensemble,0.75819,0.83673


In [18]:
importance = pd.Series(xgb_model.get_booster().get_score(importance_type="gain"))
importance.sort_values(ascending=False).head(15).round(1).to_frame("gain")

,gain
n_present_rare,98.0
loan_rev_max_start_non_fin,94.9
overdraft_app_term_max_360,86.3
loan_amount_vs_overdraft_limit,46.4
cnt_cred_loan_90,39.0
sum_deb_ul_30,36.1
overdraft_app_term_max_360_isna,33.4
cnt_deb_loan_90,29.1
cb_rate,25.2
loan_rev_min_start_fin,19.0


In [19]:
adversarial_data = pd.concat([features, test_features], ignore_index=True)
adversarial_label = np.concatenate([np.zeros(len(features)), np.ones(len(test_features))])
adversarial_model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    eval_metric="auc", tree_method="hist", enable_categorical=True, n_jobs=-1, random_state=RANDOM_STATE,
)
adversarial_prediction = cross_val_predict(
    adversarial_model, adversarial_data, adversarial_label, cv=3, method="predict_proba"
)[:, 1]
print("adversarial train-vs-test auc:", round(roc_auc_score(adversarial_label, adversarial_prediction), 4))

adversarial train-vs-test auc: 0.9101
